In [ ]:
import pickle
import numpy as np
from cuml.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os

#Datasets that will be evaluated
TASKS = [
    {'input': 'processed_BVP_data_CLASS.pkl',  'output': 'baseline_BVP_class.pkl'},
    {'input': 'processed_EDA_data_CLASS.pkl',  'output': 'baseline_EDA_class.pkl'},
    {'input': 'processed_TEMP_data_CLASS.pkl', 'output': 'baseline_TEMP_class.pkl'},
    {'input': 'processed_COMBINED-TEMP_data.pkl', 'output': 'baseline_COMBINED_3class.pkl'},
    {'input': 'processed_TEMP_data.pkl', 'output': 'baseline_TEMP.pkl'},
    {'input': 'processed_EDA_data.pkl',  'output': 'baseline_EDA.pkl'},
    {'input': 'processed_COMBINED-TEMP_data_CLASS.pkl', 'output': 'baseline_COMBINED-TEMP_class'},
    {'input': 'processed_ACC_data.pkl', 'output': 'baseline_ACC.pkl'},
    {'input': 'processed_ACC_data_CLASS.pkl', 'output': 'baseline_ACC_class.pkl'},
    {'input': 'processed_COMBINED_data.pkl', 'output': 'baseline_COMBINED.pkl'},
    {'input': 'processed_COMBINED_data_CLASS.pkl', 'output': 'baseline_COMBINED_CLASS.pkl'}
]

#Directory where the results will be stored
SAVE_DIR = 'baseline_results'

def run_logistic():
    """
    Run Leave-One-Subject-Out (LOSO) evaluation using a Logistic Regression baseline model.
    For each dataset, one subject is used as the test set while the remaining
    subjects are used for training. The final accuracy and predictions are stored 
    for each subject.
    """

    #Create output directory if it does not exist
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)

    #Loop through all datasets
    for task in TASKS:
        input_file = task['input']
        output_path = os.path.join(SAVE_DIR, task['output'])

        # Skip datasets that are not available
        if not os.path.exists(input_file):
            print(f"⏩ Skipping {input_file} (File not found)")
            continue

        # Skip datasets that have already been processed
        if os.path.exists(output_path):
            print(f"✅ Already have results for {input_file}, moving to next...")
            continue

        print(f"\n🚀 STARTING TASK: {input_file}")
        
        #Load the preprocessed dataset
        with open(input_file, 'rb') as f:
            data_dict = pickle.load(f)

        subjects = list(data_dict.keys())
        loso_results = {}
        scaler = StandardScaler()

        #Perform Leave-One-Subject-Out (LOSO) evaluation
        for test_sub in subjects:

            #Create a list of all training subjects
            train_subs = [s for s in subjects if s != test_sub]

            X_train_list, y_train_list = [], []

            #Combine the data from all training subjects
            for s in train_subs:
                X_train_list.append(data_dict[s][0])
                y_train_list.append(data_dict[s][1])
            
            #Use the remaining subject as the test set
            X_test, y_test = data_dict[test_sub]

            X_train = np.vstack(X_train_list)
            y_train = np.concatenate(y_train_list)

            #Flatten the windows so they can be used by Logistic Regression
            X_train_flat = X_train.reshape(X_train.shape[0], -1)
            X_test_flat = X_test.reshape(X_test.shape[0], -1)

            #Standardize the features
            X_train_scaled = scaler.fit_transform(X_train_flat).astype('float32')
            X_test_scaled = scaler.transform(X_test_flat).astype('float32')
            
            y_train = y_train.astype('float32')

            #Train the Logistic Regression baseline model
            model = LogisticRegression(max_iter= 2000)
            model.fit(X_train_scaled, y_train)
            
            #Generate predictions for the test subject
            preds = model.predict(X_test_scaled)

            #Convert predictions if necessary
            if hasattr(preds, 'to_numpy'):
                preds = preds.to_numpy()
            
            #Compute classification accuracy
            acc = accuracy_score(y_test, preds)

            #Store the results for the current subject
            loso_results[test_sub]= {
                'final_accuracy': acc,
                'y_true': y_test,
                'y_pred': preds
            }
            print(f"Subject {test_sub}: {acc*100:.2f}%")

            print(f"✓ Results saved to: {output_path}")

        #Save the LOSO results for the current dataset
        with open(output_path, 'wb') as f:
            pickle.dump(loso_results, f)

        #Compute the average accuracy across all subjects
        avg = np.mean([loso_results[s]['final_accuracy'] for s in subjects])
        print(f"\n✓ Finished! Average Baseline Accuracy: {avg*100:.2f}%")
        print(f"✓ Results saved to: {output_path}")
    
    
if __name__ == "__main__":
    run_logistic()  
    

ModuleNotFoundError: No module named 'cuml'